In [ ]:
"""#**3_model_training**

##**Load Feature Data**
"""

from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

train_data = spark.read.parquet("data/train_features")
test_data = spark.read.parquet("data/test_features")

"""##**Logistic Regression**"""

from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(featuresCol="features", labelCol="label", maxIter=20)
lr_model = lr.fit(train_data)

lr_model.write().overwrite().save("models/lr_model")

"""##**Naive Bayes**"""

from pyspark.ml.classification import NaiveBayes

nb = NaiveBayes(featuresCol="features", labelCol="label")
nb_model = nb.fit(train_data)

nb_model.write().overwrite().save("models/nb_model")

"""##**Random Forest**"""

from pyspark.ml.classification import RandomForestClassifier

rf = RandomForestClassifier(featuresCol="features", labelCol="label", numTrees=50)
rf_model = rf.fit(train_data)

rf_model.write().overwrite().save("models/rf_model")

"""##**SVM (OneVsRest)**"""

from pyspark.ml.classification import LinearSVC, OneVsRest

lsvc = LinearSVC(featuresCol="features", labelCol="label")
ovr = OneVsRest(classifier=lsvc)

svm_model = ovr.fit(train_data)

svm_model.write().overwrite().save("models/svm_model")

"""##**Cross Validation Example (LR)**"""

from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

evaluator = MulticlassClassificationEvaluator(metricName="accuracy")

paramGrid = ParamGridBuilder() \
    .addGrid(lr.regParam, [0.01, 0.1]) \
    .addGrid(lr.elasticNetParam, [0.0, 0.5]) \
    .build()

crossval = CrossValidator(estimator=lr,
                          estimatorParamMaps=paramGrid,
                          evaluator=evaluator,
                          numFolds=3)

cv_model = crossval.fit(train_data)
